# IRIS full-server qualification

Select a GPU runtime. Run after the eager CUDA check passes. Reuse `/content/mstar-iris` and its Python 3.12 virtualenv, or recreate them with `qualify_colab.ipynb`. This notebook adds the full serving installation and native Rust extension. GPU execution of this new server gate remains pending.

Read [SERVER.md](https://github.com/Artifex2002/mstar/blob/codex/iris/scripts/iris/SERVER.md) for acceptance criteria and troubleshooting.

## Install serving dependencies and build Rust
This can take several minutes. Rebuild after updating the checkout.

In [ ]:
%%bash
set -euo pipefail
cd /content/mstar-iris
git switch codex/iris
git pull --ff-only origin codex/iris
export PATH="$HOME/.cargo/bin:$PWD/.venv/bin:$PATH"
if ! command -v rustup >/dev/null 2>&1; then
  curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal
fi
uv pip install --python .venv/bin/python --torch-backend=auto \
  -c scripts/iris/requirements-cuda-tests.txt -e '.[iris,metrics]' maturin
env -u CONDA_PREFIX VIRTUAL_ENV="$PWD/.venv" \
  .venv/bin/maturin develop --release --locked -m rust/Cargo.toml
.venv/bin/python -c 'import mstar_rust; print(mstar_rust.__version__)'


## Run the real server checks
Choose a new output directory on each rerun. Send the report and server log even when a check fails.

In [ ]:
%%bash
set -euo pipefail
cd /content/mstar-iris
mkdir -p .ci-results/iris
OMP_NUM_THREADS=2 .venv/bin/python -m scripts.iris.qualify_server \
  --download-checkpoint --output .ci-results/iris/server-check-01 \
  2>&1 | tee .ci-results/iris/server-check-01.log


## Download the evidence
If you changed the output directory above, change it here too.

In [ ]:
import shutil

from google.colab import files

archive = shutil.make_archive(
    "/content/iris-server-check-01", "zip",
    "/content/mstar-iris/.ci-results/iris/server-check-01",
)
files.download(archive)
